# Lab 8
# Addition Attention

Aswin Singh Karki

ACE080BCT016

## Introduction

This lab report explores attention mechanisms, specifically Bahdanau (Additive) Attention and Luong (Dot-Product) Attention, within the context of sequence-to-sequence models. Traditional encoder-decoder models struggle with long input sequences due to a fixed-size context vector. Attention mechanisms address this by allowing the decoder to selectively focus on relevant parts of the input sequence during decoding, thereby improving performance for longer sentences.



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Placeholder values (These would typically be defined based on your specific model setup)
SOS_token = 0  # Start-of-sequence token
MAX_LENGTH = 10 # Maximum sequence length
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Bahdanau Attention (Additive Attention)

### Concept
Bahdanau attention, also known as additive attention, addresses the bottleneck problem in sequence-to-sequence models by allowing the decoder to dynamically focus on different parts of the encoder's output. For each output word, it computes a context vector that is a weighted sum of all encoder hidden states. The weights are determined by an alignment score between the current decoder state and each encoder hidden state, calculated using a feed-forward neural network.

### How it Works (as described in the provided text)
1.  **Encoder**: Generates annotations (hidden states `h_j`) for every word in the input sequence.
2.  **Decoder**: For each decoding step:
    *   **Score Function**: Computes attention scores `e_{t,j} = v^T tanh(W₁h_j + W₂s_t)` comparing the current decoder state (`s_t`) with all encoder hidden states (`h_j`).
    *   **Attention Weights**: Applies softmax to `e_{t,j}` to get attention weights (`α`).
    *   **Context Vector**: Creates a context vector (`c_t`) as a weighted sum of encoder hidden states using `α`.
    *   **Input to RNN**: Concatenates the context vector (`c_t`) with the embedded previous word.
    *   **Next Word Prediction**: Passes this combined vector through an RNN to generate the next target word.

In [2]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        # query: current decoder hidden state (batch_size, 1, hidden_size)
        # keys: encoder outputs (batch_size, seq_len, hidden_size)

        # Compute alignment scores
        # Wa(query) -> (batch_size, 1, hidden_size)
        # Ua(keys) -> (batch_size, seq_len, hidden_size)
        # Summing them implies broadcasting the query across the sequence length of keys
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))

        # scores: (batch_size, seq_len, 1)
        scores = scores.squeeze(2).unsqueeze(1) # (batch_size, 1, seq_len)

        # Apply softmax to get attention weights
        weights = F.softmax(scores, dim=-1) # (batch_size, 1, seq_len)

        # Compute context vector as weighted sum of keys (encoder outputs)
        context = torch.bmm(weights, keys) # (batch_size, 1, hidden_size)

        return context, weights

In [3]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        # The `hidden` tensor from RNN usually has shape (num_layers * num_directions, batch, hidden_size)
        # For attention, we need a query of shape (batch, 1, hidden_size) for the current step.
        # If hidden is (1, batch, hidden_size) for a single-layer, unidirectional RNN,
        # then query should be hidden.permute(1, 0, 2) which results in (batch, 1, hidden_size).
        query = hidden.permute(1, 0, 2) # (batch, 1, hidden_size)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2) # (batch_size, 1, embedded_size + hidden_size)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output) # (batch_size, 1, output_size)

        return output, hidden, attn_weights

## Luong Dot-Product Attention

### Concept
Luong attention, particularly the dot-product variant, is another common attention mechanism. Unlike Bahdanau attention, which calculates attention scores using a separate feed-forward network, Luong's dot-product attention computes alignment scores by simply taking the dot product between the decoder's current hidden state and each of the encoder's hidden states. It then uses these scores to create a context vector that is directly concatenated with the decoder's hidden state to form an attentional hidden state.

### How it Works (as described in the provided text)
1.  **Alignment Scores**: Computes `e_{t,i} = s_t^T h_i` by taking the dot product between the current decoder hidden state (`s_t`) and each encoder hidden state (`h_i`).
2.  **Attention Weights**: Applies softmax to the alignment scores `e_{t,i}` to get `alpha_{t,i}`.
3.  **Context Vector**: Computes the context vector `c_t` as a weighted sum of the encoder hidden states using the attention weights.
4.  **Concatenation**: Concatenates the context vector (`c_t`) with the current decoder hidden state (`s_t`).
5.  **Attentional Hidden State**: Transforms the concatenated vector using a linear layer and a tanh activation to produce the attentional hidden state `s~_t = tanh(W_c[c_t;s_t])`.

In [4]:
class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)

            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        # query: (batch_size, 1, hidden_size)
        # keys.transpose(1, 2): (batch_size, hidden_size, seq_len)
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        ) # (batch_size, 1, seq_len)

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1) # (batch_size, 1, seq_len)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        # weights: (batch_size, 1, seq_len)
        # keys: (batch_size, seq_len, hidden_size)
        context = torch.bmm(
            weights,
            keys
        ) # (batch_size, 1, hidden_size)

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        # context: (batch_size, 1, hidden_size)
        # query: (batch_size, 1, hidden_size)
        combined = torch.cat(
            (context, query),
            dim=-1
        ) # (batch_size, 1, 2 * hidden_size)

        # Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        ) # (batch_size, 1, hidden_size)

        return attentional_hidden, weights

## Resources

Here are some key theoretical resources related to the attention mechanisms discussed in this lab:

*   **Bahdanau Attention (Additive Attention)**:
    *   [Neural Machine Translation by Jointly Learning to Align and Translate (Bahdanau et al., 2014)](https://arxiv.org/abs/1409.0473)

*   **Luong Attention (Dot-Product Attention)**:
    *   [Effective Approaches to Attention-based Neural Machine Translation (Luong et al., 2015)](https://arxiv.org/abs/1508.04025)

These papers introduce and detail the respective attention mechanisms, providing the theoretical foundation for their implementation.

## Demonstration of Attention Mechanisms

To see these attention mechanisms in action, we'll create some dummy input tensors for `query` (decoder hidden state) and `keys` (encoder outputs) and pass them through our defined attention classes. This will show the `context` vector and the `attention weights` they produce.

In [5]:
# Define a hidden size for our dummy example
hidden_size = 256
seq_len = 5 # Length of the encoder output sequence
batch_size = 2

# Create dummy encoder outputs (keys) and decoder query
dummy_encoder_outputs = torch.randn(batch_size, seq_len, hidden_size).to(device)
dummy_decoder_query = torch.randn(batch_size, 1, hidden_size).to(device)

print(f"Dummy Encoder Outputs shape: {dummy_encoder_outputs.shape}")
print(f"Dummy Decoder Query shape: {dummy_decoder_query.shape}")

Dummy Encoder Outputs shape: torch.Size([2, 5, 256])
Dummy Decoder Query shape: torch.Size([2, 1, 256])


### Bahdanau Attention Output

In [6]:
# Instantiate Bahdanau Attention
bahdanau_attn = BahdanauAttention(hidden_size).to(device)

# Get context and weights
b_context, b_weights = bahdanau_attn(dummy_decoder_query, dummy_encoder_outputs)

print("Bahdanau Attention Outputs:")
print(f"  Context Vector shape: {b_context.shape}")
print(f"  Attention Weights shape: {b_weights.shape}")
print("  Sample Context Vector (first batch, first item):\n", b_context[0, 0, :5].detach().cpu().numpy()) # Display first 5 elements
print("  Sample Attention Weights (first batch):\n", b_weights[0].detach().cpu().numpy())

Bahdanau Attention Outputs:
  Context Vector shape: torch.Size([2, 1, 256])
  Attention Weights shape: torch.Size([2, 1, 5])
  Sample Context Vector (first batch, first item):
 [-1.3018992 -0.6384768  0.5148116 -0.3285833  0.5407185]
  Sample Attention Weights (first batch):
 [[0.15454455 0.19173273 0.25891206 0.19349535 0.2013153 ]]


### Luong Dot-Product Attention Output

In [7]:
# Instantiate Luong Dot-Product Attention
luong_attn = LuongDotAttention(hidden_size).to(device)

# Get attentional hidden state and weights
l_attentional_hidden, l_weights = luong_attn(dummy_decoder_query, dummy_encoder_outputs)

print("Luong Dot-Product Attention Outputs:")
print(f"  Attentional Hidden State shape: {l_attentional_hidden.shape}")
print(f"  Attention Weights shape: {l_weights.shape}")
print("  Sample Attentional Hidden State (first batch, first item):\n", l_attentional_hidden[0, 0, :5].detach().cpu().numpy()) # Display first 5 elements
print("  Sample Attention Weights (first batch):\n", l_weights[0].detach().cpu().numpy())

Luong Dot-Product Attention Outputs:
  Attentional Hidden State shape: torch.Size([2, 1, 256])
  Attention Weights shape: torch.Size([2, 1, 5])
  Sample Attentional Hidden State (first batch, first item):
 [-0.57381636  0.30236024  0.32042563 -0.2641009  -0.890304  ]
  Sample Attention Weights (first batch):
 [[1.7613722e-08 1.0000000e+00 5.1652712e-20 7.0569364e-13 4.0876282e-11]]


### Interpretation of Outputs

*   **Context Vector (Bahdanau)** / **Attentional Hidden State (Luong)**: These are the outputs that incorporate information from the encoder states, weighted by attention. This new vector is then typically used by the decoder to predict the next word.
*   **Attention Weights**: These are probability distributions over the encoder's input sequence. Each value indicates how much 'attention' the decoder pays to a specific part of the source sequence when generating the current target word. The sum of weights for each `query` (decoder step) will be 1.